# Segmentation
## Overview
### Task definition
- Assign a class label to every pixel (dense prediction)
- Output is a label map aligned with the image grid
- Tensor view: Input `B x C x H x W`, logits Z `B x K x H x W` for K classes
### Classification vs detection vs segmentation
- Classification: Output 1 label / image, low structure, mostly use accuracy as metric
- Detection: Output boxes + classes, medium structure, mostly use mAP as metric
- Segmentation: Output mask/masks, high structure, use mIoU, Dice, PQ as metrics
- Segmentation couples appearance with spatial layout
### Semantic segmentation
- One category per pixel (no object identity)
- Output: class index map Y K x H x W (per image)
- Usage: Scene parsing, tissue labels, land-cover maps
### Instance segmentation
- Seperate objects of same class (which instance)
- Output often a set of masks with class ids and scores
- Grouping and duplicate suppression matter (have detection-like errors)
# Panoptic segmentation
- Things (countable instance) + stuff (amorphous regions)
- Non-overlapping regions covering all pixels
- Panoptic Quality PQ is decomposed into SQ x RQ
- Segmentation Quality measures mask overlap, Recognition Quality measures recognition/matching, PQ measures both
- SQ: average mask overlap over matched pairs = sum(IoU(p, g)) / |TP|
- RQ: recognition/matching quality = |TP| / (|TP| +|FP|/ / 2 + |FN| / 2)
- PQ = SQ x RQ
### Segmentation usage
- Medical: organ/tumor boundaries drive clinical decisions
- Driving/robotics: free space, lanes, obstacles
- Satellite/industry: parcels, defects, material regions
- Failures are often boundary and small-object driven, not only wrong class
### Logits and probabilities
Per-pixel logits z_i,k, softmax gives class probabilities: p_i,k = exp(z_i,k) / sum(exp(z_i,c))
- Training minimizes loss on logits/probabilties
- Deployment often uses argmax_k p_i,k for class maps
## Encoder-Decoder Architecture - UNet
### Dense prediction pipeline - level 0
X -> Backbone -> (Neck) -> Head -> Z, Z (B, K, H, W)
- Backbone: feature extraction
- Neck: refinement/fusion/adaption before head
- Head: task projection, mostly 1x1 Conv2d with K filters
### Dense prediction pipeline - level 2
Input -> Encoder1 -> Encoder2 -> Encoder3 -> Bottleneck -> Decoder1 -> Decoder2 -> Decoder3 -> Neck -> Head -> Logits (Skip connection for respectitive encoder decoder pair)
- Backbone now enclose encoder + bottleneck + decoder + skip links
- Skip connections inject high resolution encoder maps into decoder stages (sharp boundaries)
- Classical UNet: neck is identity/absent; modern: necks can be refinement, ASPP, attention, fusion, boundary refinement
### Compare classification & segmentation backbone
- Classification backbone is typically encoder-only with global pooling: image -> encoder -> pooled vector -> FC
- Segmentation backbone must preserve/reconstruct spatial structure: image -> encoder -> bottleneck -> decoder -> dense map
### UNet - generalized segmentation backbone
- Backbone_UNet = Encoder + Bottleneck + Decoder + Skip Connections
- Backbone_UNet = Backbone_Classfication + Bridge/Bottleneck + Decoder + Skip Connections
- Encoder can be custom convolution blocks or pretrained ResNet/EfficientNet/ConvNeXt/ViT/Swin
- Decoder adapts encoder features for dense prediction
- Skip connections recover localization details
### Neck - feature refinement
F_neck = g_phi(F_backbone), Z = Conv_1x1(F_neck)
- Optional
- Refines/fuses/adapts/enriches dense feature maps
+ Identity (classical UNet)
+ Conv neck (one/more Conv2D block)
+ ASSPP neck (DeepLab/ASPP context)
+ Attention neck (SE/CBAM/Transformer attention)
+ Fusion neck (FPN/BiFPN/SegFormer MLP Fusion)
+ Boundary neck (edge refinement module)
### Head != Decoder
- Decoder reconstructs spatial feature maps
- Neck refines reconstructed maps 
- Head projects channels to task output: F (B, C, H, W), Head(F) = Conv2d(C, K, 1)(F) = Z (B, K, H, W)
- Small but determines output semantics
### Segmentation checklist
- Encoder/visual backbone
- Recover spatial resolution method
- Skip connection/multi-scale function used
- Neck refinement
- Head output format
- Loss & metric define success
- Most papers modify backbone, neck, head, loss, training protocol
### Mental model
- Basic UNet: Input -> UNet Backbone -> Head
- Practical modern segmentation: Input -> Visual Encoder -> Decoder -> Neck (optional) -> Head
- Research view: Input -> Backbone -> Neck -> Task Head -> Losses/Metrics
### Why not plain CNN classification?
- Global pooling destroys spatial features
- Segmentation needs per-pixel decisions with localization cues
- Need a pathway preserves/reconstructs spatial detail
### Encoder
1. Conv blocks
- Stacked conv + nonlinearity + norm
- Learn local textures and semantic patterns
2. Downsampling
- Pooling/strided conv reduces (H, W) and expands receptive field
- Produces a feature hierarchy: fine -> coarse
### Bottlenecl
- Deepest, coarsest map with strongest context
- Often widest channels. highest compute per pixel (fewer pixels)
### Decoder
1. Upsampling
- Increase resolution toward input size (transpose conv, interpolate + conv, ...)
- Without skip, upsampling alone tends to produce blurry masks
2. Feature fusion
- Combine coarse semantics with finer spatial cues
- UNet uses skip connection from encoder to decoder
### Skip connections
- Pooling loses high-frequency edges and thin structures
- Skip route localization information directly to the decoder
- Skips are the main mechanism for sharp boundaries in CNN UNets
### Concatenation vs addition
- Concat: Keep encoder and decoder features distinct (UNet common)
- Add: requires matched channels, cheaper but can cause blur roles
- Concat: Safer when boundary detail is critical
### UNet engineering view
- Explicit split: context (deep) + localization (skips)
- Strong inductive bias for grid-structured dense outputs
### Memory footprint
- VRAM grows with batch, resolution, skip channels, decoder width
- Patch training, mixed precision, smaller base channels help
- Hits memory limits before accuracy limits
### Segmentation head
- Applied after decoder/neck
- Typically Conv 1x1 -> K logits per pixel
### Outputs
- Multi-class (K > 2): softmax along class dimension
- Binary: sigmoid per pixel
- Hard labels: y_i = argmax_k p_i,k
